# **Data and Information Quality Project**

In [1]:
import pandas as pd
from ydata_profiling import ProfileReport
import numpy as np
import json
import os
import re
import matplotlib.pyplot as plt
import Levenshtein as lev
import pyphonetics
from pyphonetics import Soundex
import jaro
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
import recordlinkage
from recordlinkage.datasets import load_febrl1

In [2]:
DATASET = pd.read_csv('dataset/Comune-di-Milano-Servizi-alla-persona-parrucchieri-estetisti(in).csv',sep=';',encoding='unicode_escape')


In [3]:
DATASET

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,ZD,Prevalente,Superficie altri usi,Superficie lavorativa
0,NaN,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0
1,NaN,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0
2,NaN,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0
3,NaN,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN
4,NaN,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0
...,...,...,...,...,...,...,...,...,...,...
3904,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA SARPI FRA' PAOLO N. 1 con ingr.da v.le mon...,VIA,SARPI FRA' PAOLO,1,7210.0,1,NaN,NaN,NaN
3905,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,CSO DI PORTA TICINESE N. 4 ; (z.d. 1),CSO,DI PORTA TICINESE,4,541.0,1,NaN,NaN,65.0
3906,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN
3907,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,NaN,640.0,1,NaN,NaN,NaN


# *Data Quality Assesement*


In [4]:
#Data information
DATASET.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3909 entries, 0 to 3908
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tipo esercizio pa      3878 non-null   object 
 1   Ubicazione             3909 non-null   object 
 2   Tipo via               3908 non-null   object 
 3   Via                    3908 non-null   object 
 4   Civico                 3832 non-null   object 
 5   Codice via             3908 non-null   float64
 6   ZD                     3908 non-null   object 
 7   Prevalente             294 non-null    object 
 8   Superficie altri usi   745 non-null    float64
 9   Superficie lavorativa  2601 non-null   float64
dtypes: float64(3), object(7)
memory usage: 305.5+ KB


##### Single column analisys

In [5]:
# Distinct values for each column
DATASET.nunique()

Tipo esercizio pa         103
Ubicazione               3554
Tipo via                   17
Via                      1370
Civico                    235
Codice via               1376
ZD                         10
Prevalente                 63
Superficie altri usi       52
Superficie lavorativa     145
dtype: int64

In [6]:
# Uniqueness percentage for each column
UNIQUENESS = (DATASET.nunique() / DATASET.shape[0]) * 100
UNIQUENESS

Tipo esercizio pa         2.634945
Ubicazione               90.918393
Tipo via                  0.434894
Via                      35.047327
Civico                    6.011768
Codice via               35.200819
ZD                        0.255820
Prevalente                1.611665
Superficie altri usi      1.330263
Superficie lavorativa     3.709389
dtype: float64

In [7]:
#Information about the type of esercizio
DATASET.value_counts("Tipo esercizio pa")

Tipo esercizio pa
Parrucchiere per signora                                         1048
ACCONCIATORE                                                      586
Parrucchiere per uomo                                             439
TIPO A - REG.2003                                                 335
TIPO A - REG.2003;TIPO B CENTRO DI ABBRONZATURA                   313
                                                                 ... 
TIPO A-B-C-D;ACCONCIATORE                                           1
TIPO A-B-C-D;Acconciatore                                           1
TIPO A-B-C-D;Estetista in profumeria                                1
TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ESTETICI DIMAGRIM       1
Truccatore                                                          1
Name: count, Length: 103, dtype: int64

##### Completeness

In [8]:
# For each column
NULL_VALUES = DATASET.isnull().sum()
NOT_NULL_VALUES = DATASET.notnull().sum()
ROWS = DATASET.shape[0]
COMPLETENESS = NOT_NULL_VALUES / ROWS
COMPLETENESS = COMPLETENESS.map('{:.2%}'.format)
COMPLETENESS

Tipo esercizio pa         99.21%
Ubicazione               100.00%
Tipo via                  99.97%
Via                       99.97%
Civico                    98.03%
Codice via                99.97%
ZD                        99.97%
Prevalente                 7.52%
Superficie altri usi      19.06%
Superficie lavorativa     66.54%
dtype: object

In [9]:
# For the entire dataset
TOT_NULL_VALUES = DATASET.isnull().sum().sum() 
TOT_NOT_NULL_VALUES = DATASET.notnull().sum().sum()
TOT_COMPLETENESS = TOT_NOT_NULL_VALUES / DATASET.size
TOT_COMPLETENESS = '{:.2%}'.format(TOT_COMPLETENESS)
TOT_COMPLETENESS

'79.03%'

##### Duplication

In [10]:
DATASET.duplicated().any()

np.True_

In [11]:
DATASET[DATASET.duplicated()]

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,ZD,Prevalente,Superficie altri usi,Superficie lavorativa
88,Acconciatore,VIA CORREGGIO N. 8 (z.d. 7),VIA,CORREGGIO,8,6287.0,7,ACCONCIATORE,NaN,NaN


# *Data Profiling*


In [12]:
profile = ProfileReport(DATASET, title=" Report comune di Milano Servizi alla persona di parrucchieri e estetisti")
profile.to_file("profilingReport/Report comune di Milano Servizi alla persona di parrucchieri e estetisti.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 125.43it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# *Data Wrangling*

Renaming and Sorting

In [13]:
DATASET.rename(columns={"Tipo esercizio pa":"Tipo esercizio", "Prevalente":"Attivita Primaria", "ZD":"Municipio"},inplace=True)

In [14]:
DATASET = DATASET.sort_values(by = ['Tipo esercizio', "Ubicazione"], ascending=True)
DATASET.head()

,Tipo esercizio,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attivita Primaria,Superficie altri usi,Superficie lavorativa
33,(z.d. 9),CSO,COMO,15,1111,9.0,ACCONCIATORE,NaN,195.0,NaN
208,ACCONCIATORE,ALZ NAVIGLIO PAVESE N. 52 ; (z.d. 6),ALZ,NAVIGLIO PAVESE,52,5161.0,6,NaN,2.0,12.0
209,ACCONCIATORE,BST DI PORTA VOLTA N. 13 ; (z.d. 1),BST,DI PORTA VOLTA,13,1066.0,1,NaN,NaN,NaN
211,ACCONCIATORE,CSO BUENOS AIRES N. 64 ; (z.d. 3),CSO,BUENOS AIRES,64,2129.0,3,NaN,NaN,57.0
212,ACCONCIATORE,CSO COLOMBO CRISTOFORO N. 8 ; (z.d. 6),CSO,COLOMBO CRISTOFORO,8,5114.0,6,NaN,NaN,41.0


Standardization

In [15]:
def to_upper_safe(x):
    if isinstance(x, str):
        return x.upper()
    return x

DATASET = DATASET.map(lambda x: to_upper_safe(x) if pd.notnull(x) else x)

In [16]:
# Transform "Tipo Esercizio"

new_cols = DATASET["Tipo esercizio"].str.split(";", expand=True)
new_cols = new_cols.apply(lambda col: col.str.strip())
new_cols = new_cols.replace("", np.nan)
new_cols = new_cols.replace({None: np.nan})
new_cols.columns = [f"Tipo_esercizio_{i+1}" for i in range(new_cols.shape[1])]

DATASET = pd.concat([DATASET, new_cols], axis=1)

split_cols = DATASET[[c for c in DATASET.columns if c.startswith("Tipo_esercizio_")]]

acconciatori = {"ACCONCIATORE", "PARRUCCHIERE MISTO", "PARRUCCHIERE PER SIGNORA", "PARRUCCHIERE PER UOMO", "BARBIERE"}
estetisti = {"ESTETISTA", "ESTETISTA IN PROFUMERIA", "TIPO A - REG.2003", "TIPO A ESTETICA MANUALE", "TIPO A-B-C-D", "MANICURE", "PEDICURE ESTETICO", "TRUCCATORE"}
centri_abbronzatura = {"CENTRO ABBRONZATURA", "TIPO B CENTRO DI ABBRONZATURA", "CENTRO BENESSERE", "CENTRO MASSAGGI", "TIPO A-B-C-D"}
trattamenti = {"TIPO C TRATT.ESTETICI DIMAGRIM", "TIPO D ESTET.APPAR.ELETTROMECC", "TIPO A-B-C-D"}
tatuaggi = {"ESECUZIONE DI TATUAGGI E PIERCING"}

def has_any_from_group(group_set):
    return split_cols.isin(group_set).any(axis=1)

DATASET["ACCONCIATORE"] = has_any_from_group(acconciatori)
DATASET["ESTETISTA"] = has_any_from_group(estetisti)
DATASET["CENTRO ABBRONZATURA"] = has_any_from_group(centri_abbronzatura)
DATASET["TRATTAMENTO"] = has_any_from_group(trattamenti)
DATASET["TATUAGGI E PIERCING"] = has_any_from_group(tatuaggi)

DATASET = DATASET.drop(columns=split_cols.columns)
DATASET.head()

,Tipo esercizio,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attivita Primaria,Superficie altri usi,Superficie lavorativa,ACCONCIATORE,ESTETISTA,CENTRO ABBRONZATURA,TRATTAMENTO,TATUAGGI E PIERCING
33,(Z.D. 9),CSO,COMO,15,1111,9.0,ACCONCIATORE,NaN,195.0,NaN,False,False,False,False,False
208,ACCONCIATORE,ALZ NAVIGLIO PAVESE N. 52 ; (Z.D. 6),ALZ,NAVIGLIO PAVESE,52,5161.0,6,NaN,2.0,12.0,True,False,False,False,False
209,ACCONCIATORE,BST DI PORTA VOLTA N. 13 ; (Z.D. 1),BST,DI PORTA VOLTA,13,1066.0,1,NaN,NaN,NaN,True,False,False,False,False
211,ACCONCIATORE,CSO BUENOS AIRES N. 64 ; (Z.D. 3),CSO,BUENOS AIRES,64,2129.0,3,NaN,NaN,57.0,True,False,False,False,False
212,ACCONCIATORE,CSO COLOMBO CRISTOFORO N. 8 ; (Z.D. 6),CSO,COLOMBO CRISTOFORO,8,5114.0,6,NaN,NaN,41.0,True,False,False,False,False


In [17]:
# Check types
DATASET.dtypes

Tipo esercizio            object
Ubicazione                object
Tipo via                  object
Via                       object
Civico                    object
Codice via               float64
Municipio                 object
Attivita Primaria         object
Superficie altri usi     float64
Superficie lavorativa    float64
ACCONCIATORE                bool
ESTETISTA                   bool
CENTRO ABBRONZATURA         bool
TRATTAMENTO                 bool
TATUAGGI E PIERCING         bool
dtype: object

Column Splitting

In [18]:
# Split "Ubicazione"
col = "Ubicazione" 

pattern = r"""
^\s*
(?P<tipo_via_to_check>\S+)
\s+
(?P<via_to_check>.*?)
\s+N\.\s*
(?P<civico_to_check>\S+)
(?:.*?Z\.D\.\s*(?P<municipio_to_check>\d+))?
"""

parsed = DATASET[col].str.extract(pattern, flags=re.VERBOSE)

DATASET = pd.concat([DATASET, parsed], axis=1)
DATASET.head()

,Tipo esercizio,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attivita Primaria,Superficie altri usi,Superficie lavorativa,ACCONCIATORE,ESTETISTA,CENTRO ABBRONZATURA,TRATTAMENTO,TATUAGGI E PIERCING,tipo_via_to_check,via_to_check,civico_to_check,municipio_to_check
33,(Z.D. 9),CSO,COMO,15,1111,9.0,ACCONCIATORE,NaN,195.0,NaN,False,False,False,False,False,NaN,NaN,NaN,NaN
208,ACCONCIATORE,ALZ NAVIGLIO PAVESE N. 52 ; (Z.D. 6),ALZ,NAVIGLIO PAVESE,52,5161.0,6,NaN,2.0,12.0,True,False,False,False,False,ALZ,NAVIGLIO PAVESE,52,6
209,ACCONCIATORE,BST DI PORTA VOLTA N. 13 ; (Z.D. 1),BST,DI PORTA VOLTA,13,1066.0,1,NaN,NaN,NaN,True,False,False,False,False,BST,DI PORTA VOLTA,13,1
211,ACCONCIATORE,CSO BUENOS AIRES N. 64 ; (Z.D. 3),CSO,BUENOS AIRES,64,2129.0,3,NaN,NaN,57.0,True,False,False,False,False,CSO,BUENOS AIRES,64,3
212,ACCONCIATORE,CSO COLOMBO CRISTOFORO N. 8 ; (Z.D. 6),CSO,COLOMBO CRISTOFORO,8,5114.0,6,NaN,NaN,41.0,True,False,False,False,False,CSO,COLOMBO CRISTOFORO,8,6


# *Error Detection & Correction*

In [19]:
# Fix "Ubicazione"

DATASET['Civico'] = DATASET.apply(
    lambda row: row['civico_to_check'] if pd.notna(row['civico_to_check']) else row['Civico'],
    axis=1
)

DATASET['Municipio'] = DATASET.apply(
    lambda row: row['municipio_to_check'] if pd.notna(row['municipio_to_check']) else row['Municipio'],
    axis=1
)

DATASET['Via'] = DATASET.apply(
    lambda row: row['via_to_check'] if pd.notna(row['via_to_check']) else row['Via'],
    axis=1
)

DATASET['Tipo via'] = DATASET.apply(
    lambda row: row['tipo_via_to_check'] if pd.notna(row['tipo_via_to_check']) else row['Tipo via'],
    axis=1
)

DATASET = DATASET.drop(columns=['civico_to_check', 'municipio_to_check', 'via_to_check', 'tipo_via_to_check'])
DATASET = DATASET.drop(columns=['Ubicazione'])

In [20]:
# Fix "Attivita primaria"
tipo_bool_cols = ["ACCONCIATORE", "ESTETISTA", "CENTRO ABBRONZATURA", "TRATTAMENTO", "TATUAGGI E PIERCING"]
tipo_bool = DATASET[tipo_bool_cols]

has_true = tipo_bool.any(axis=1)

first_true_col = tipo_bool.idxmax(axis=1)

first_true_col = first_true_col.where(has_true, np.nan)

DATASET["Attivita Primaria"] = first_true_col

# *Null Values Handling*

In [21]:
# Drop rows where "Attivita Primaria" and "Tipo esercizio" are null
DATASET = DATASET.dropna(subset=["Attivita Primaria", "Tipo esercizio"], how='all')

DATASET = DATASET.drop(columns="Tipo esercizio", errors="ignore")

In [22]:
# Fill "Superficie lavorativa" with median value grouping by "Tipo esercizio"
group_median = (
    DATASET
    .groupby(tipo_bool_cols)["Superficie lavorativa"]
    .transform("median") 
)

mask_superficie_null = DATASET["Superficie lavorativa"].isna()
DATASET.loc[mask_superficie_null, "Superficie lavorativa"] = group_median[mask_superficie_null]

In [23]:
# Fill "Superficie altri usi" with 0
DATASET["Superficie altri usi"] = DATASET["Superficie altri usi"].fillna(0)

In [24]:
# FIll "Civico" with 0
# Better to know at least a part of the address than not having the row at all
DATASET['Civico'] = DATASET['Civico'].fillna('NOT-AVAILABLE')

In [25]:
# Dropping possible remaining null values (if any they are for sure in columns where imputation was impossible)
DATASET = DATASET.dropna()

# *Outlier Detection*

In [26]:
# Columns are either categorical or objects.
# The only columns with numeical values are "Superficie lavorativa", "Superficie altri usi" and "Codice via"
# which are not suited for outlier detection.

# *Duplicate Detection*

In [27]:
# Drop exact duplicates
DATASET = DATASET.drop_duplicates()

In [28]:
# Use 'Sorted Neighbourhood' to find candidate pairs
DATASET["SN_key"] = DATASET["Tipo via"] + " " + DATASET["Via"]

indexer = recordlinkage.index.SortedNeighbourhood(
    on="SN_key",
    window=25
)

candidate_links = indexer.index(DATASET)

C:\Users\leona\AppData\Local\Temp\ipykernel_27408\106055086.py:4: DeprecationWarning: The argument 'on' is deprecated. Use 'left_on=...' and 'right_on=None' to simulate the behaviour of 'on'.
  indexer = recordlinkage.index.SortedNeighbourhood(


In [29]:
# Compare records
compare_cl = recordlinkage.Compare()

compare_cl.exact("Tipo via", "Tipo via", label="tipo_via_match")
compare_cl.string("Via", "Via", method="jarowinkler", label="via_sim")
compare_cl.string("Civico", "Civico", method="jarowinkler", threshold=0.75 ,label="civico_sim")
compare_cl.exact("Municipio", "Municipio", label="municipio_match")
compare_cl.exact("Codice via", "Codice via", label="codice_via_match")
compare_cl.string("Attivita Primaria", "Attivita Primaria", method="jarowinkler", threshold=0.75, label="attivita_sim")

for col in ["Via", "Civico", "Attivita Primaria"]:
    DATASET[col] = DATASET[col].astype(str)

features = compare_cl.compute(candidate_links, DATASET)


In [30]:
# Compare matches

features['score'] = features.sum(axis=1)

# theoretical max score = 6
T = 5.5

print("Scores distribution:")
print(features["score"].describe())

matches = features[features['score'] >= T]

print("Number of couples classified as possible duplicates:", len(matches))
print(matches.head())

Scores distribution:
count    148471.000000
mean          2.455583
std           0.914857
min           0.000000
25%           1.612573
50%           2.524074
75%           2.664573
max           6.000000
Name: score, dtype: float64
Number of couples classified as possible duplicates: 530
         tipo_via_match  via_sim  civico_sim  municipio_match  \
248 249               1      1.0         1.0                1   
400 401               1      1.0         1.0                1   
429 406               1      1.0         1.0                1   
438 437               1      1.0         1.0                1   
455 454               1      1.0         1.0                1   

         codice_via_match  attivita_sim  score  
248 249                 1           1.0    6.0  
400 401                 1           1.0    6.0  
429 406                 1           1.0    6.0  
438 437                 1           1.0    6.0  
455 454                 1           1.0    6.0  


In [31]:
# Drop candidate duplicates

indices_to_drop = set(matches.index.get_level_values(1))

print("Numero di righe droppate dal DATASET:", len(indices_to_drop))

matches_reset = matches.reset_index()
bool_cols = ["ACCONCIATORE", "ESTETISTA", "CENTRO ABBRONZATURA", "TRATTAMENTO", "TATUAGGI E PIERCING"]

for idx, row in matches_reset.iterrows():
    idx0 = int(row['level_0'])
    idx1 = int(row['level_1'])
    for col in bool_cols:
        DATASET.loc[idx0, col] = DATASET.loc[idx0, col] | DATASET.loc[idx1, col]

DATASET_dedup = DATASET.drop(index=indices_to_drop).copy()

DATASET = DATASET_dedup

if "SN_key" in DATASET.columns:
    DATASET = DATASET.drop(columns=["SN_key"])

Numero di righe droppate dal DATASET: 444


# *Results*

In [32]:
#Data information
DATASET.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3403 entries, 208 to 1105
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tipo via               3403 non-null   object 
 1   Via                    3403 non-null   object 
 2   Civico                 3403 non-null   object 
 3   Codice via             3403 non-null   float64
 4   Municipio              3403 non-null   object 
 5   Attivita Primaria      3403 non-null   object 
 6   Superficie altri usi   3403 non-null   float64
 7   Superficie lavorativa  3403 non-null   float64
 8   ACCONCIATORE           3403 non-null   bool   
 9   ESTETISTA              3403 non-null   bool   
 10  CENTRO ABBRONZATURA    3403 non-null   bool   
 11  TRATTAMENTO            3403 non-null   bool   
 12  TATUAGGI E PIERCING    3403 non-null   bool   
dtypes: bool(5), float64(3), object(5)
memory usage: 255.9+ KB


In [33]:
DATASET.duplicated().any()

np.False_

In [34]:
profile = ProfileReport(DATASET, title=" Report comune di Milano Servizi alla persona di parrucchieri e estetisti")
profile.to_file("profilingReport/Report cleaned comune di Milano Servizi alla persona di parrucchieri e estetisti.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 13/13 [00:00<00:00, 1362.16it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [35]:
DATASET.to_csv("dataset/cleaned_Comune-di-Milano-Servizi-alla-persona-parrucchieri-estetisti(in).csv")